# 🌀 RealNVP

In this notebook, we'll walk through the steps required to train your own RealNVP network to predict the distribution of a demo dataset

The code has been adapted from the excellent [RealNVP tutorial](https://keras.io/examples/generative/real_nvp) created by Mandolini Giorgio Maria, Sanna Daniele and Zannini Quirini Giorgio available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2

# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# TensorFlow Probability (tfp) -> torch.distributions, built into PyTorch
import torch.distributions as D

# Select the device - PyTorch needs to be told where to run
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
COUPLING_DIM = 256
COUPLING_LAYERS = 2
INPUT_DIM = 2
REGULARIZATION = 0.01
BATCH_SIZE = 256
EPOCHS = 300

In [ ]:
# Load the data
data = datasets.make_moons(30000, noise=0.05)[0].astype("float32")
data = torch.tensor(data)

# Equivalent of Keras layers.Normalization().adapt(): standardise each
# feature to zero mean and unit variance using statistics of the data
normalized_data = (data - data.mean(dim=0)) / data.std(dim=0, unbiased=False)

plt.scatter(normalized_data[:, 0], normalized_data[:, 1], c="green")
plt.show()

## 2. Build the RealNVP network <a name="build"></a>

In [ ]:
class Coupling(nn.Module):
    """Two small MLPs that predict the scale (s) and translation (t)."""

    def __init__(self, input_dim, coupling_dim):
        super().__init__()

        def mlp(final_activation):
            return nn.Sequential(
                nn.Linear(input_dim, coupling_dim),
                nn.ReLU(),
                nn.Linear(coupling_dim, coupling_dim),
                nn.ReLU(),
                nn.Linear(coupling_dim, coupling_dim),
                nn.ReLU(),
                nn.Linear(coupling_dim, coupling_dim),
                nn.ReLU(),
                nn.Linear(coupling_dim, input_dim),
                final_activation,
            )

        self.s = mlp(nn.Tanh())  # tanh keeps the log-scale small and stable
        self.t = mlp(nn.Identity())  # "linear" activation

    def forward(self, x):
        return self.s(x), self.t(x)


# Note on REGULARIZATION: the Keras version attaches an l2 kernel_regularizer to
# every Dense layer, but its custom train_step only minimises log_loss, so those
# penalties never reach the gradients. We therefore leave them out here.

In [ ]:
class RealNVP(nn.Module):
    def __init__(self, input_dim, coupling_layers, coupling_dim, regularization):
        super().__init__()
        self.coupling_layers = coupling_layers
        self.regularization = regularization  # unused, see the note above

        # Buffers are saved with the model and moved by .to(device),
        # but are not trained (unlike nn.Parameter)
        self.register_buffer("loc", torch.zeros(input_dim))
        self.register_buffer("scale_diag", torch.ones(input_dim))
        self.register_buffer(
            "masks",
            torch.tensor([[0, 1], [1, 0]] * (coupling_layers // 2), dtype=torch.float32),
        )
        # nn.ModuleList (not a plain list) so PyTorch registers the sub-networks
        self.layers_list = nn.ModuleList(
            [Coupling(input_dim, coupling_dim) for i in range(coupling_layers)]
        )

    @property
    def distribution(self):
        # Same as tfp.distributions.MultivariateNormalDiag: independent
        # Normals, with log_prob summed over the last dimension
        return D.Independent(D.Normal(self.loc, self.scale_diag), 1)

    def forward(self, x, training=True):
        # training=True:  data -> latent (f, used for the loss)
        # training=False: latent -> data (g, used for generation)
        log_det_inv = 0
        direction = 1
        if training:
            direction = -1
        for i in range(self.coupling_layers)[::direction]:
            x_masked = x * self.masks[i]
            reversed_mask = 1 - self.masks[i]
            s, t = self.layers_list[i](x_masked)
            s = s * reversed_mask
            t = t * reversed_mask
            gate = (direction - 1) / 2
            x = (
                reversed_mask
                * (x * torch.exp(direction * s) + direction * t * torch.exp(gate * s))
                + x_masked
            )
            log_det_inv = log_det_inv + gate * torch.sum(s, dim=1)
        return x, log_det_inv

    def log_loss(self, x):
        # Change of variables: log p(x) = log p_Z(f(x)) + log|det df/dx|
        y, logdet = self(x)
        log_likelihood = self.distribution.log_prob(y) + logdet
        return -torch.mean(log_likelihood)


model = RealNVP(
    input_dim=INPUT_DIM,
    coupling_layers=COUPLING_LAYERS,
    coupling_dim=COUPLING_DIM,
    regularization=REGULARIZATION,
).to(device)

print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Train the RealNVP network <a name="train"></a>

In [ ]:
# Keras compile() -> just create the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
writer = SummaryWriter(log_dir="./logs")  # TensorBoard logging


class ImageGenerator:
    """Plain Python version of the Keras callback; the training loop calls on_epoch_end."""

    def __init__(self, num_samples):
        self.num_samples = num_samples

    @torch.no_grad()  # no gradients needed to generate
    def generate(self):
        model.eval()
        # From data to latent space.
        z, _ = model(normalized_data.to(device))

        # From latent space to data.
        samples = model.distribution.sample((self.num_samples,))
        x, _ = model(samples, training=False)

        return x.cpu(), z.cpu(), samples.cpu()

    def display(self, x, z, samples, save_to=None):
        f, axes = plt.subplots(2, 2)
        f.set_size_inches(8, 5)

        axes[0, 0].scatter(
            normalized_data[:, 0], normalized_data[:, 1], color="r", s=1
        )
        axes[0, 0].set(title="Data space X", xlabel="x_1", ylabel="x_2")
        axes[0, 0].set_xlim([-2, 2])
        axes[0, 0].set_ylim([-2, 2])
        axes[0, 1].scatter(z[:, 0], z[:, 1], color="r", s=1)
        axes[0, 1].set(title="f(X)", xlabel="z_1", ylabel="z_2")
        axes[0, 1].set_xlim([-2, 2])
        axes[0, 1].set_ylim([-2, 2])
        axes[1, 0].scatter(samples[:, 0], samples[:, 1], color="g", s=1)
        axes[1, 0].set(title="Latent space Z", xlabel="z_1", ylabel="z_2")
        axes[1, 0].set_xlim([-2, 2])
        axes[1, 0].set_ylim([-2, 2])
        axes[1, 1].scatter(x[:, 0], x[:, 1], color="g", s=1)
        axes[1, 1].set(title="g(Z)", xlabel="x_1", ylabel="x_2")
        axes[1, 1].set_xlim([-2, 2])
        axes[1, 1].set_ylim([-2, 2])

        plt.subplots_adjust(wspace=0.3, hspace=0.6)
        if save_to:
            plt.savefig(save_to)
            print(f"\nSaved to {save_to}")

        plt.show()

    def on_epoch_end(self, epoch):
        if epoch % 10 == 0:
            x, z, samples = self.generate()
            self.display(
                x,
                z,
                samples,
                save_to="./output/generated_img_%03d.png" % (epoch),
            )


img_generator_callback = ImageGenerator(num_samples=3000)

In [ ]:
train_loader = DataLoader(normalized_data, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for x in train_loader:
        x = x.to(device)
        loss = model.log_loss(x)

        optimizer.zero_grad()  # reset gradients from the previous step
        loss.backward()  # backpropagation
        optimizer.step()  # update the weights

        total_loss += loss.item() * len(x)

    epoch_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {epoch_loss:.4f}")
    writer.add_scalar("loss", epoch_loss, epoch)
    img_generator_callback.on_epoch_end(epoch)

writer.close()

## 4. Generate images <a name="generate"></a>

In [ ]:
x, z, samples = img_generator_callback.generate()

In [ ]:
img_generator_callback.display(x, z, samples)